<a href="https://colab.research.google.com/github/sugargeeks/dashboard-bi-kmeans-kinerja-akademik-siswa/blob/main/Coding_TA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install streamlit scikit-learn pandas numpy plotly openpyxl xlrd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 69.8 MB/s eta 0:00:00


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

Selecting previously unselected package cloudflared.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.9.3) ...
Setting up cloudflared (2026.9.3) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [ ]:
!cloudflared --version

cloudflared version 2026.9.3 (built 2026-09-24-16:07 UTC)


In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


# ==========================================================
# KONFIGURASI
# ==========================================================

st.set_page_config(
    page_title="Klasterisasi Akademik Siswa",
    page_icon="🎓",
    layout="wide"
)


# ==========================================================
# JUDUL
# ==========================================================

st.title("🎓 Dashboard Klasterisasi Akademik Siswa")

st.markdown(
    """
    Aplikasi ini melakukan klasterisasi siswa menggunakan
    algoritma **K-Means Clustering** berdasarkan rata-rata
    nilai mata pelajaran.
    """
)


# ==========================================================
# SIDEBAR
# ==========================================================

st.sidebar.header("⚙️ Pengaturan")

fase = st.sidebar.selectbox(
    "Pilih Fase",
    [
        "Fase A",
        "Fase B/C",
    ]
)

uploaded_file = st.sidebar.file_uploader(
    "Upload Dataset Excel",
    type=["xlsx", "xls"]
)



jumlah_cluster = st.sidebar.selectbox(
    "Jumlah Cluster (K)",
    options=[2, 3],
    index=1
)

st.sidebar.info(
    f"Jumlah cluster yang digunakan: K = {jumlah_cluster}"
)

# ==========================================================
# DAFTAR MATA PELAJARAN
# ==========================================================

mapel_fase_a = {
    "Agama": [
        "Agama_harian",
        "Agama_PTS",
        "Agama_PAS"
    ],

    "PKN": [
        "PKN_harian",
        "PKN_PTS",
        "PKN_PAS"
    ],

    "Bahasa Indonesia": [
        "B.indo_harian",
        "B.indo_PTS",
        "B.indo_PAS"
    ],

    "Matematika": [
        "MTK_harian",
        "MTK_PTS",
        "MTK_PAS"
    ],

    "SBDP": [
        "SBDP_harian",
        "SBDP_PTS",
        "SBDP_PAS"
    ],

    "PJOK": [
        "PJOK_harian",
        "PJOK_PTS",
        "PJOK_PAS"
    ],

    "Mulok": [
        "Mulok_harian",
        "Mulok_PTS",
        "Mulok_PAS"
    ]
}


mapel_ipas = {
    "IPAS": [
        "IPAS_harian",
        "IPAS_PTS",
        "IPAS_PAS"
    ]
}


# ==========================================================
# PILIH MAPEL BERDASARKAN FASE
# ==========================================================

if fase == "Fase A":

    kelompok_mapel = mapel_fase_a.copy()

else:

    kelompok_mapel = mapel_fase_a.copy()
    kelompok_mapel.update(mapel_ipas)


# ==========================================================
# FUNGSI LOAD DATA
# ==========================================================

@st.cache_data
def load_data(file):

    df = pd.read_excel(file)

    # Bersihkan nama kolom
    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
    )

    # Hapus kolom No jika ada
    if "No" in df.columns:
        df = df.drop(columns=["No"])

    # Hapus baris kosong
    df = df.dropna(how="all")

    return df


# ==========================================================
# FUNGSI CLEANING
# ==========================================================

def cleaning_data(df):

    data = df.copy()

    kolom_nilai = []

    for mapel, kolom in kelompok_mapel.items():

        for col in kolom:

            if col in data.columns:

                kolom_nilai.append(col)

                data[col] = pd.to_numeric(
                    data[col],
                    errors="coerce"
                )

    # Missing value → median
    for col in kolom_nilai:

        if data[col].isna().sum() > 0:

            median = data[col].median()

            data[col] = data[col].fillna(median)

    return data


# ==========================================================
# FUNGSI FEATURE ENGINEERING
# ==========================================================

def create_features(df):

    data = df.copy()

    fitur = []

    for mapel, kolom in kelompok_mapel.items():

        kolom_valid = [
            col
            for col in kolom
            if col in data.columns
        ]

        if len(kolom_valid) > 0:

            nama_fitur = f"Rata_{mapel}"

            data[nama_fitur] = (
                data[kolom_valid]
                .mean(axis=1)
            )

            fitur.append(nama_fitur)

    if len(fitur) > 0:

        data["Rata_Rata_Nilai"] = (
            data[fitur]
            .mean(axis=1)
        )

    return data, fitur


# ==========================================================
# FUNGSI LABEL CLUSTER
# ==========================================================

def create_cluster_label(df, fitur):

    rata = (
        df.groupby("Cluster")[fitur]
        .mean()
        .mean(axis=1)
        .sort_values()
    )

    urutan = rata.index.tolist()

    label = {}

    if len(urutan) == 2:

        label = {
            urutan[0]: "Nilai Rendah",
            urutan[1]: "Nilai Tinggi"
        }

    elif len(urutan) == 3:

        label = {
            urutan[0]: "Nilai Rendah",
            urutan[1]: "Nilai Sedang",
            urutan[2]: "Nilai Tinggi"
        }

    else:

        for i, cluster in enumerate(urutan):

            label[cluster] = f"Cluster {i + 1}"

    df["Kategori"] = df["Cluster"].map(label)

    return df


# ==========================================================
# HALAMAN AWAL
# ==========================================================

if uploaded_file is None:

    st.info(
        "📂 Upload dataset Excel melalui sidebar "
        "untuk memulai analisis."
    )

    st.markdown(
        """
        ### Mata Pelajaran

        **Fase A**
        - Agama
        - PKN
        - Bahasa Indonesia
        - Matematika
        - SBDP
        - PJOK
        - Mulok

        **Fase B dan C**
        - Agama
        - PKN
        - Bahasa Indonesia
        - Matematika
        - SBDP
        - PJOK
        - Mulok
        - IPAS
        """
    )

    st.stop()


# ==========================================================
# LOAD DATA
# ==========================================================

df = load_data(uploaded_file)


# ==========================================================
# DATASET
# ==========================================================

st.header("📁 1. Dataset")

col1, col2, col3 = st.columns(3)

with col1:
    st.metric(
        "Jumlah Siswa",
        len(df)
    )

with col2:
    st.metric(
        "Jumlah Kolom",
        len(df.columns)
    )

with col3:
    st.metric(
        "Fase",
        fase
    )


st.dataframe(
    df.head(10),
    use_container_width=True
)


# ==========================================================
# CLEANING
# ==========================================================

st.header("🧹 2. Pembersihan Data")

df_clean = cleaning_data(df)

st.success(
    "Pembersihan data berhasil dilakukan."
)


# ==========================================================
# FEATURE ENGINEERING
# ==========================================================

st.header("📐 3. Pembentukan Fitur")

df_fitur, fitur_mapel = create_features(
    df_clean
)


if len(fitur_mapel) < 2:

    st.error(
        "Kolom nilai mata pelajaran yang diperlukan "
        "tidak ditemukan."
    )

    st.write("Kolom dataset:")

    st.write(
        list(df.columns)
    )

    st.stop()


st.write(
    "Fitur yang digunakan:"
)

st.write(
    fitur_mapel
)


st.dataframe(
    df_fitur[fitur_mapel + ["Rata_Rata_Nilai"]].head(10),
    use_container_width=True
)


# ==========================================================
# DATA MODEL
# ==========================================================

X = df_fitur[
    fitur_mapel
].copy()


# Standardisasi
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)


# ==========================================================
# ELBOW METHOD
# ==========================================================

st.header("📈 4. Evaluasi Jumlah Cluster")

st.subheader("Elbow Method")

inertia = []

maks_k = min(
    3,
    len(df_fitur) - 1
)

for k in range(2, maks_k + 1):

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    model.fit(X_scaled)

    inertia.append(
        model.inertia_
    )


fig, ax = plt.subplots(
    figsize=(8, 4)
)

ax.plot(
    range(2, maks_k + 1),
    inertia,
    marker="o"
)

ax.set_xlabel("Jumlah Cluster (K)")
ax.set_ylabel("Inertia")
ax.set_title("Elbow Method")

ax.grid(True)

st.pyplot(fig)



# ==========================================================
# SILHOUETTE SCORE
# ==========================================================

st.subheader(
    "Silhouette Score"
)

silhouette_scores = []

for k in range(2, maks_k + 1):

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = model.fit_predict(
        X_scaled
    )

    score = silhouette_score(
        X_scaled,
        labels
    )

    silhouette_scores.append(score)


fig, ax = plt.subplots(
    figsize=(8, 4)
)

ax.plot(
    range(2, maks_k + 1),
    silhouette_scores,
    marker="o"
)

ax.set_xlabel(
    "Jumlah Cluster (K)"
)

ax.set_ylabel(
    "Silhouette Score"
)

ax.set_title(
    "Silhouette Score"
)

ax.grid(True)

st.pyplot(fig)


k_terbaik = (
    list(range(2, maks_k + 1))
    [
        np.argmax(silhouette_scores)
    ]
)


st.info(
    f"💡 Berdasarkan Silhouette Score, "
    f"K terbaik adalah **{k_terbaik}**."
)


# ==========================================================
# K-MEANS
# ==========================================================

st.header(" 5. Klasterisasi K-Means")

kmeans = KMeans(
    n_clusters=jumlah_cluster,
    random_state=42,
    n_init=10
)

cluster = kmeans.fit_predict(
    X_scaled
)


df_hasil = df_fitur.copy()

df_hasil["Cluster"] = cluster


# Label cluster
df_hasil = create_cluster_label(
    df_hasil,
    fitur_mapel
)


# ==========================================================
# SILHOUETTE SCORE K YANG DIPILIH
# ==========================================================

score_final = silhouette_score(
    X_scaled,
    cluster
)


col1, col2 = st.columns(2)

with col1:

    st.metric(
        "Jumlah Cluster",
        jumlah_cluster
    )

with col2:

    st.metric(
        "Silhouette Score",
        round(score_final, 3)
    )


# ==========================================================
# DISTRIBUSI CLUSTER
# ==========================================================

st.subheader(
    "Distribusi Siswa per Cluster"
)

distribusi = (
    df_hasil
    .groupby(
        ["Cluster", "Kategori"]
    )
    .size()
    .reset_index(
        name="Jumlah Siswa"
    )
)


st.dataframe(
    distribusi,
    use_container_width=True
)


fig, ax = plt.subplots(
    figsize=(8, 5)
)

sns.barplot(
    data=distribusi,
    x="Kategori",
    y="Jumlah Siswa",
    ax=ax
)

ax.set_title(
    "Distribusi Siswa Berdasarkan Cluster"
)

ax.set_xlabel(
    "Kategori Cluster"
)

ax.set_ylabel(
    "Jumlah Siswa"
)

plt.xticks(rotation=20)

st.pyplot(fig)

# ==========================================================
# VISUALISASI HASIL KLASTERISASI
# ==========================================================

st.subheader("Visualisasi Hasil Klasterisasi")

# Pilihan mata pelajaran
col_x, col_y = st.columns(2)

with col_x:
    fitur_x = st.selectbox(
        "Pilih Mata Pelajaran Sumbu X",
        options=fitur_mapel,
        format_func=lambda x: x.replace("Rata_", "")
    )

with col_y:
    fitur_y = st.selectbox(
        "Pilih Mata Pelajaran Sumbu Y",
        options=fitur_mapel,
        index=1 if len(fitur_mapel) > 1 else 0,
        format_func=lambda x: x.replace("Rata_", "")
    )


# ==========================================================
# SCATTER PLOT
# ==========================================================

fig, ax = plt.subplots(
    figsize=(10, 6)
)

sns.scatterplot(
    data=df_hasil,
    x=fitur_x,
    y=fitur_y,
    hue="Kategori",
    style="Kategori",
    s=100,
    ax=ax
)

ax.set_title(
    "Visualisasi Hasil Klasterisasi K-Means"
)

ax.set_xlabel(
    fitur_x.replace("Rata_", "Rata-rata ")
)

ax.set_ylabel(
    fitur_y.replace("Rata_", "Rata-rata ")
)

ax.grid(True, alpha=0.3)

ax.legend(
    title="Kategori"
)

st.pyplot(fig)

# ==========================================================
# PROFIL CLUSTER
# ==========================================================

st.header(
    "📊 6. Profil Setiap Cluster"
)

profil = (
    df_hasil
    .groupby(
        ["Cluster", "Kategori"]
    )[fitur_mapel]
    .mean()
    .round(2)
)


st.dataframe(
    profil,
    use_container_width=True
)


# ==========================================================
# HEATMAP
# ==========================================================

st.subheader(
    "Heatmap Profil Cluster"
)

heatmap_data = (
    df_hasil
    .groupby("Kategori")[fitur_mapel]
    .mean()
)


fig, ax = plt.subplots(
    figsize=(12, 6)
)

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".1f",
    cmap="YlGnBu",
    ax=ax
)

ax.set_title(
    "Rata-Rata Nilai Mata Pelajaran "
    "Berdasarkan Cluster"
)

st.pyplot(fig)


# ==========================================================
# HASIL SISWA
# ==========================================================

st.header(
    "👨‍🎓 7. Hasil Klasterisasi Siswa"
)

# ============================================================
# FILTER KELAS & CLUSTER
# ============================================================

col_filter1, col_filter2 = st.columns(2)

with col_filter1:
    pilihan_kelas = st.multiselect(
        "🏫 Filter Kelas",
        options=sorted(df_hasil["Kelas"].dropna().unique()),
        default=sorted(df_hasil["Kelas"].dropna().unique()),
        key="filter_kelas"
    )

with col_filter2:
    pilihan_cluster = st.multiselect(
        "🔵 Filter Cluster",
        options=sorted(df_hasil["Cluster"].dropna().unique()),
        default=sorted(df_hasil["Cluster"].dropna().unique()),
        key="filter_cluster"
    )


# ============================================================
# TERAPKAN FILTER
# ============================================================

df_filtered = df_hasil.copy()

if pilihan_kelas:
    df_filtered = df_filtered[
        df_filtered["Kelas"].isin(pilihan_kelas)
    ]

if pilihan_cluster:
    df_filtered = df_filtered[
        df_filtered["Cluster"].isin(pilihan_cluster)
    ]


# ============================================================
# KOLOM TABEL
# ============================================================

kolom_identitas = []

for col in [
    "NISN",
    "NIS",
    "Nama",
    "Kelas"
]:
    if col in df_filtered.columns:
        kolom_identitas.append(col)


kolom_tampil = (
    kolom_identitas
    + fitur_mapel
    + [
        "Rata_Rata_Nilai",
        "Cluster",
        "Kategori"
    ]
)


# ============================================================
# TABEL HASIL
# ============================================================

st.dataframe(
    df_filtered[kolom_tampil]
    .sort_values("Cluster"),
    use_container_width=True
)

# ==========================================================
# DOWNLOAD
# ==========================================================

st.header(
    "📥 8. Download Hasil"
)


hasil_download = df_hasil[
    kolom_tampil
].copy()


csv = hasil_download.to_csv(
    index=False
).encode("utf-8-sig")


st.download_button(
    label="⬇️ Download Hasil Klasterisasi CSV",
    data=csv,
    file_name=(
        f"hasil_klasterisasi_{fase.replace(' ', '_')}.csv"
    ),
    mime="text/csv"
)

Overwriting app.py


In [ ]:
#Memberhentikan proses yang sedang berjalan

!pkill -f streamlit || true
!pkill -f cloudflared || true

^C
^C


In [ ]:
#Jalankan Streamlit
!nohup streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > streamlit.log 2>&1 &

In [ ]:
!curl -I http://localhost:8501

HTTP/1.1 200 OK
date: Sun, 04 Oct 2026 06:29:09 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 6463
last-modified: Sun, 04 Oct 2026 06:17:27 GMT
etag: "10ea0bd574ad1f59f70a9e86b1e69689"
cache-control: no-cache



In [ ]:
!rm -f cloudflared.log

In [ ]:
#Jalankan Cloudflare Tunnel
!nohup cloudflared tunnel --url http://localhost:8501 > cloudflared.log 2>&1 &


In [ ]:
import time
import re
import os

for i in range(30):
    time.sleep(1)

    if os.path.exists("cloudflared.log"):
        with open("cloudflared.log", "r") as f:
            log = f.read()

        urls = re.findall(
            r'https://[-a-z0-9]+\.trycloudflare\.com',
            log
        )

        if urls:
            print("==========================================")
            print("URL CLOUDFLARE BARU")
            print("==========================================")
            print(urls[-1])
            break
else:
    print("URL belum ditemukan.")
    print(log)

URL CLOUDFLARE BARU
https://sip-votes-rating-latino.trycloudflare.com
